# Taxonomy Extraction

Run the enriched taxonomy prompt on papers predicted as POSITIVE by the classifier.
Uses the `taxonomy` task config from `llm_labeller.py`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path().resolve().parent))

In [2]:
DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"

In [3]:
from src.labelling.llm_labeller import claude_batch_submit, claude_batch_results

## 1. Load positive predictions

In [4]:
predictions = pd.read_parquet(
    DATA_DIR / "predictions" / "inference_predictions_base.parquet"
)

# Keep only predicted positives
positives = predictions[predictions["predicted"] == 1].copy()
print(f"Positive predictions: {len(positives):,} papers")


Positive predictions: 3,738 papers


In [6]:
anthology = pd.read_parquet(DATA_DIR / "raw" / "anthology_enriched.parquet")

# Filter to positives in the anthology
positives_paper = anthology[anthology["bibkey"].isin(positives["bibkey"])]
positives_paper = positives_paper[positives_paper["year"] == 2019][:30]
print(f"Positive predictions in anthology: {len(positives_paper):,} papers")

Positive predictions in anthology: 30 papers


## 2. Submit Claude batch (taxonomy task)

In [7]:
job_id = claude_batch_submit(positives_paper, task="taxonomy")
job_id

'msgbatch_01TjPCA71T6MmaaUj2fTMBEq'

In [7]:
job_id = "msgbatch_01BE6FuJZqzPcAZfBzy7hYFV"

## 3. Retrieve results

In [9]:
results = claude_batch_results(job_id, positives_paper, task="taxonomy")


In [ ]:
results.to_parquet(DATA_DIR / "labeled" / "taxonomy_claude_1.parquet", index=False)
results.head()